In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML
from sklearn.metrics import mean_absolute_error, r2_score

print("--- TAHAP 1: PERSIAPAN DATA OTOMATIS ---")
# Memuat dataset (tidak memerlukan pengunduhan eksternal)
housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

# Pembagian data (80% Pelatihan, 20% Pengujian)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Data pelatihan: {X_train.shape[0]} baris.")
print(f"Data pengujian: {X_test.shape[0]} baris.\n")

print("--- TAHAP 2: BENCHMARKING MASSAL (LAZYPREDICT) ---")
# Eksekusi otomatis puluhan algoritma untuk mencari nilai dasar (baseline)
reg_benchmark = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None)
models, _ = reg_benchmark.fit(X_train, X_test, y_train, y_test)

print("Top 3 Model Baseline (Tanpa Optimasi Lanjutan):")
print(models[['R-Squared', 'RMSE', 'Time Taken']].head(3))
print("\n")

print("--- TAHAP 3: OPTIMASI OTOMATIS PENUH (FLAML AutoML) ---")
# Memerintahkan mesin untuk mengeksplorasi arsitektur terbaik selama 60 detik
automl_engine = AutoML()
settings = {
    "time_budget": 60,       # 60 detik waktu pencarian
    "metric": 'r2',          # Optimasi ditargetkan pada R2
    "task": 'regression',    # Tugas regresi
    "log_file_name": 'housing_automl.log',
    "verbose": 0
}

automl_engine.fit(X_train=X_train, y_train=y_train, **settings)

print("Hasil Eksekusi AutoML:")
print(f"Algoritma Pemenang: {automl_engine.best_estimator}")
print(f"Hyperparameter Terbaik: {automl_engine.best_config}\n")

print("--- TAHAP 4: EVALUASI AKHIR MODEL PEMENANG ---")
# Prediksi menggunakan model yang telah dioptimasi secara otomatis
prediksi_final = automl_engine.predict(X_test)

# Perhitungan metrik evaluasi
mae_final = mean_absolute_error(y_test, prediksi_final)
r2_final = r2_score(y_test, prediksi_final)

print("Performa Model AutoML pada Data Pengujian (Data Baru):")
print(f"R2 Score : {r2_final:.4f} (Mendekati 1.0 semakin baik)")
print(f"MAE      : {mae_final:.4f} (Deviasi rata-rata harga rumah dalam ratusan ribu USD)")

# Menampilkan perbandingan 5 data aktual vs prediksi
df_komparasi = pd.DataFrame({
    'Harga Aktual (Ratusan Ribu USD)': y_test.values[:5],
    'Prediksi AutoML': np.round(prediksi_final[:5], 3)
})
print("\nSampel Prediksi vs Aktual:")
print(df_komparasi)

--- TAHAP 1: PERSIAPAN DATA OTOMATIS ---
Data pelatihan: 16512 baris.
Data pengujian: 4128 baris.

--- TAHAP 2: BENCHMARKING MASSAL (LAZYPREDICT) ---
Top 3 Model Baseline (Tanpa Optimasi Lanjutan):
                               R-Squared      RMSE  Time Taken
Model                                                         
LGBMRegressor                   0.838610  0.459877    0.319138
HistGradientBoostingRegressor   0.835544  0.464225    0.775410
XGBRegressor                    0.830137  0.471794    0.220669


--- TAHAP 3: OPTIMASI OTOMATIS PENUH (FLAML AutoML) ---
Hasil Eksekusi AutoML:
Algoritma Pemenang: xgb_limitdepth
Hyperparameter Terbaik: {'n_estimators': 226, 'max_depth': 9, 'min_child_weight': 5.647460375037914, 'learning_rate': 0.05385568510222151, 'subsample': 0.8413048297641477, 'colsample_bylevel': 0.7310705843442475, 'colsample_bytree': 0.8092314042000436, 'reg_alpha': 0.001058358871316641, 'reg_lambda': 0.036942259210126616}

--- TAHAP 4: EVALUASI AKHIR MODEL PEMENANG ---